In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Configuration
# ============================================================
INPUT_FILE  = 'analysis_dataset.csv'
OUTPUT_FILE = 'ai_scenario_results.csv'
PLOT_FILE   = 'ai_scenario_gradient.png'

# Beta coefficient from M2 (main OLS model)
# mortality_rate ~ log(density_rad+1) + rucc_2023 + income_10k + pct_uninsured_adults
BETA = -2.2383

# AI capacity scenarios: additive increments (ΔAI per 100,000 population)
# Anchored to observed density distribution among nonzero counties:
#   Low      = 25th percentile of nonzero counties = 1.27
#   Moderate = median of nonzero counties          = 5.17
#   High     = 75th percentile of nonzero counties = 10.92
# These represent hypothetical policy targets for AI-enabled capacity expansion.
SCENARIOS = {
    'Baseline (ΔAI = 0)':      0.0,
    'Low (ΔAI = 1.3)':         1.2716,
    'Moderate (ΔAI = 5.2)':    5.1688,
    'High (ΔAI = 10.9)':       10.9232,
}

# Shortage subgroup definitions
# Primary:   extreme shortage (density = 0)
# Robustness: severe shortage (density < 25th percentile of nonzero counties)
EXTREME_SHORTAGE_CUTOFF  = 0.0
SEVERE_SHORTAGE_CUTOFF   = 1.2716

# Continuous gradient range
GRADIENT_MAX  = 10.9232
GRADIENT_STEPS = 200

# ============================================================
# Step 1: Load data
# ============================================================
print("Loading analysis dataset...")
df = pd.read_csv(INPUT_FILE, low_memory=False, dtype={'FIPS': str})

# Restrict to counties with complete data (same as OLS sample)
df_main = df[
    df['mortality_rate'].notna() &
    df['rucc_2023'].notna() &
    df['median_household_income'].notna() &
    df['pct_uninsured_adults'].notna()
].copy()

print(f"Analysis sample: {len(df_main):,} counties")
print(f"Extreme shortage (density=0):  {(df_main['density_radiologist']==0).sum():,} counties")
print(f"Severe shortage (density<1.27): {(df_main['density_radiologist']<SEVERE_SHORTAGE_CUTOFF).sum():,} counties")

# ============================================================
# Step 2: Define AI capacity injection function
# ============================================================
def compute_mortality_change(density, delta_ai, beta=BETA):
    """
    Compute predicted mortality rate change under AI capacity injection.

    Mechanism: AI adds ΔAI radiologists per 100,000 population (additive).
    This allows zero-density counties to benefit from AI deployment,
    consistent with teleradiology and AI-enabled diagnostic services.

    density_new = density + ΔAI
    Δlog = log(density_new + 1) - log(density + 1)
    Δmortality = β × Δlog
    """
    log_density_original = np.log(density + 1)
    log_density_new      = np.log(density + delta_ai + 1)
    delta_log            = log_density_new - log_density_original
    delta_mortality      = beta * delta_log
    return delta_mortality

# ============================================================
# Step 3: Scenario analysis — all counties
# ============================================================
print("\n=== Scenario Analysis: All Counties ===")
results = []

for scenario_name, delta_ai in SCENARIOS.items():
    df_main[f'delta_mortality_{scenario_name}'] = compute_mortality_change(
        df_main['density_radiologist'], delta_ai
    )
    mean_change = df_main[f'delta_mortality_{scenario_name}'].mean()
    median_change = df_main[f'delta_mortality_{scenario_name}'].median()
    print(f"\n{scenario_name}")
    print(f"  Mean mortality change:   {mean_change:.4f} per 100,000")
    print(f"  Median mortality change: {median_change:.4f} per 100,000")

# ============================================================
# Step 4: Shortage subgroup analysis
# ============================================================
print("\n=== Shortage Subgroup Analysis ===")

subgroups = {
    'Extreme shortage (density = 0)':      df_main['density_radiologist'] == 0,
    'Severe shortage (density < 1.27)':    df_main['density_radiologist'] < SEVERE_SHORTAGE_CUTOFF,
    'Non-shortage (density >= 1.27)':      df_main['density_radiologist'] >= SEVERE_SHORTAGE_CUTOFF,
}

scenario_summary = []

for subgroup_name, mask in subgroups.items():
    subgroup = df_main[mask].copy()
    print(f"\n{subgroup_name} — N = {len(subgroup):,}")
    print(f"  Mean baseline mortality: {subgroup['mortality_rate'].mean():.2f}")

    for scenario_name, delta_ai in SCENARIOS.items():
        delta_mort = compute_mortality_change(
            subgroup['density_radiologist'], delta_ai
        )
        mean_change  = delta_mort.mean()
        total_deaths = mean_change * subgroup['population_2021'].sum() / 100_000
        print(f"  {scenario_name}: Δmortality = {mean_change:.4f} "
              f"(≈ {abs(total_deaths):,.0f} deaths averted per year)")

        scenario_summary.append({
            'subgroup':        subgroup_name,
            'scenario':        scenario_name,
            'delta_ai':        delta_ai,
            'n_counties':      len(subgroup),
            'mean_baseline_mortality': subgroup['mortality_rate'].mean().round(4),
            'mean_delta_mortality':    round(mean_change, 4),
            'estimated_deaths_averted': round(abs(total_deaths), 1)
        })

results_df = pd.DataFrame(scenario_summary)

# ============================================================
# Step 5: Continuous gradient sensitivity analysis
# ============================================================
print("\n=== Continuous Gradient Sensitivity Analysis ===")
gradient_values = np.linspace(0, GRADIENT_MAX, GRADIENT_STEPS)

gradient_results = {}
for subgroup_name, mask in subgroups.items():
    subgroup = df_main[mask].copy()
    mean_changes = []
    for delta_ai in gradient_values:
        delta_mort = compute_mortality_change(
            subgroup['density_radiologist'], delta_ai
        )
        mean_changes.append(delta_mort.mean())
    gradient_results[subgroup_name] = mean_changes

print("Gradient analysis complete.")
print(f"Range: ΔAI = 0 to {GRADIENT_MAX:.1f}")
print(f"Steps: {GRADIENT_STEPS}")

# ============================================================
# Step 6: Plot continuous gradient (publication-ready version)
# Changes:
#   1. Y-axis flipped to positive values (mortality REDUCTION)
#   2. Extreme shortage = thick solid red; Severe = dashed orange
#   3. Policy-driven title
#   4. ΔAI=0 vertical line removed (redundant)
#   5. Annotation highlighting shortage vs non-shortage gap
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

# Plot lines: extreme=thick solid, severe=dashed, non-shortage=solid
plot_styles = {
    'Extreme shortage (density = 0)':   dict(color='#d62728', linewidth=2.5, linestyle='-'),
    'Severe shortage (density < 1.27)': dict(color='#ff7f0e', linewidth=1.5, linestyle='--'),
    'Non-shortage (density >= 1.27)':   dict(color='#1f77b4', linewidth=2.0, linestyle='-'),
}

for subgroup_name, mean_changes in gradient_results.items():
    # Flip sign: negative change → positive reduction
    reductions = [-x for x in mean_changes]
    ax.plot(gradient_values, reductions,
            label=subgroup_name,
            **plot_styles[subgroup_name])

# Mark discrete scenario points (skip ΔAI=0)
scenario_line_colors = {'Low (ΔAI = 1.3)': 'green',
                        'Moderate (ΔAI = 5.2)': 'purple',
                        'High (ΔAI = 10.9)': 'red'}
for scenario_name, delta_ai in SCENARIOS.items():
    if delta_ai == 0:
        continue
    color = scenario_line_colors[scenario_name]
    ax.axvline(x=delta_ai, color=color, linestyle='--', alpha=0.4, linewidth=1)
    ax.text(delta_ai + 0.1, 0.1, f'ΔAI={delta_ai:.1f}',
            fontsize=8, color=color, rotation=90, va='bottom')

# Annotation: highlight gap at ΔAI=5.2
delta_ref = 5.1688
idx_ref = np.argmin(np.abs(gradient_values - delta_ref))

shortage_val  = -gradient_results['Extreme shortage (density = 0)'][idx_ref]
nonshortage_val = -gradient_results['Non-shortage (density >= 1.27)'][idx_ref]

ax.annotate('', xy=(delta_ref, shortage_val), xytext=(delta_ref, nonshortage_val),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text(delta_ref + 0.2, (shortage_val + nonshortage_val) / 2,
        f'Shortage: {shortage_val:.1f}\nNon-shortage: {nonshortage_val:.1f}',
        fontsize=8.5, color='black', va='center')

ax.set_xlabel('AI Capacity Injection (ΔAI, radiologists per 100,000 population)', fontsize=12)
ax.set_ylabel('Reduction in Cancer Mortality Rate\n(per 100,000 population)', fontsize=12)
ax.set_title('Greater Mortality Reduction from AI in Radiologist-Shortage Counties', fontsize=13)
ax.legend(fontsize=10, loc='upper left')
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_FILE, dpi=150, bbox_inches='tight')
print(f"\nPlot saved to: {PLOT_FILE}")

# ============================================================
# Step 7: Save results
# ============================================================
results_df.to_csv(OUTPUT_FILE, index=False)
print(f"Results saved to: {OUTPUT_FILE}")

print("\n" + "="*60)
print("KEY FINDINGS SUMMARY")
print("="*60)
extreme = results_df[results_df['subgroup'] == 'Extreme shortage (density = 0)']
for _, row in extreme.iterrows():
    print(f"{row['scenario']}: "
          f"Δmortality = {row['mean_delta_mortality']:.4f}, "
          f"≈ {row['estimated_deaths_averted']:,.0f} deaths averted/year")

print("\nDONE. Proceed to Script 6 (spatial regression robustness check).")

Loading analysis dataset...
Analysis sample: 3,073 counties
Extreme shortage (density=0):  1,195 counties
Severe shortage (density<1.27): 1,659 counties

=== Scenario Analysis: All Counties ===

Baseline (ΔAI = 0)
  Mean mortality change:   0.0000 per 100,000
  Median mortality change: -0.0000 per 100,000

Low (ΔAI = 1.3)
  Mean mortality change:   -1.1268 per 100,000
  Median mortality change: -1.5873 per 100,000

Moderate (ΔAI = 5.2)
  Mean mortality change:   -2.6776 per 100,000
  Median mortality change: -3.6884 per 100,000

High (ΔAI = 10.9)
  Mean mortality change:   -3.8237 per 100,000
  Median mortality change: -5.1239 per 100,000

=== Shortage Subgroup Analysis ===

Extreme shortage (density = 0) — N = 1,195
  Mean baseline mortality: 167.98
  Baseline (ΔAI = 0): Δmortality = 0.0000 (≈ 0 deaths averted per year)
  Low (ΔAI = 1.3): Δmortality = -1.8365 (≈ 380 deaths averted per year)
  Moderate (ΔAI = 5.2): Δmortality = -4.0726 (≈ 842 deaths averted per year)
  High (ΔAI = 10.9